In [1]:
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg as la
from undi import MuonNuclearInteraction

# --- Physical Constants & Geometry Setup ---
angtom = 1e-10
r_nn = 1.172 * angtom  # F-mu bond length in CaF2 (from paper)


def build_f_mu_f_cluster(include_nnn=True):
    """
    Builds the F-mu-F cluster geometry in CaF2 with optional 8 NNN fluorines.
    """
    neighbors = [
        # Muon at origin
        {"Position": np.array([0.0, 0.0, 0.0]), "Label": "mu"},
        # 2 Nearest-Neighbor Fluorines along z-axis
        {"Position": np.array([0.0, 0.0, r_nn]), "Label": "F"},
        {"Position": np.array([0.0, 0.0, -r_nn]), "Label": "F"},
    ]

    if include_nnn:
        # 8 Next-Nearest-Neighbor Fluorines in CaF2 fluorite structure
        # Distances are scaled by zeta = 0.937 (accounting for distortions)
        zeta = 0.937
        a = 5.451 * angtom  # Lattice parameter
        nnn_pos = (a / 2.0) * zeta * np.array([
            [0.5, 0.5, 0.5], [0.5, -0.5, 0.5], [-0.5, 0.5, 0.5], [-0.5, -0.5, 0.5],
            [0.5, 0.5, -0.5], [0.5, -0.5, -0.5], [-0.5, 0.5, -0.5], [-0.5, -0.5, -0.5]
        ])
        for pos in nnn_pos:
            neighbors.append({"Position": pos, "Label": "F"})

    return neighbors


def von_neumann_entropy(rho: np.ndarray, eps: float = 1e-12) -> float:
    """Calculates von Neumann entropy S = -Tr(rho log2(rho)) in bits."""
    evals = la.eigvalsh(rho)
    evals = evals[evals > eps]
    return float(-np.sum(evals * np.log2(evals)))


def compute_signals_and_entropy(
    neighbors, tlist, muon_init_axis="z", orientation=np.array([0.0, 0.0, 1.0])
):
    """
    Computes time evolution of P(t), S_mu(t), and S_F(t) using exact diagonalization.
    """
    ns = MuonNuclearInteraction(deepcopy(neighbors), log_level="warning")
    if orientation.ndim == 1:
        ns.translate_rotate_sample_vec(orientation)
    else:
        ns.translate_rotate_sample(orientation)

    H = ns.H  # Extract Hamiltonian built by UNDI
    dim_total = H.shape[0]
    dim_mu = 2
    dim_env = dim_total // dim_mu

    # Initial muon state |psi_mu>
    if muon_init_axis == "x":
        psi_mu = np.array([1.0, 1.0], dtype=np.complex128) / np.sqrt(2)
        sigma_op = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=np.complex128)
    else:  # 'z'
        psi_mu = np.array([1.0, 0.0], dtype=np.complex128)
        sigma_op = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=np.complex128)

    # Initial density matrix rho(0) = |psi_mu><psi_mu| (x) (I_env / dim_env)
    rho_mu_0 = np.outer(psi_mu, np.conj(psi_mu))
    rho_env_0 = np.eye(dim_env, dtype=np.complex128) / dim_env
    rho_0 = np.kron(rho_mu_0, rho_env_0)

    # Exact Diagonalization
    evals, evecs = la.eigh(H)
    rho_diag = evecs.conj().T @ rho_0 @ evecs
    E_diff = evals[:, None] - evals[None, :]

    O_mu = np.kron(sigma_op, np.eye(dim_env, dtype=np.complex128))

    P_t = np.zeros(len(tlist))
    S_mu = np.zeros(len(tlist))
    S_F = np.zeros(len(tlist))

    for idx, t in enumerate(tlist):
        phase_factors = np.exp(-1j * E_diff * t)
        rho_t = evecs @ (rho_diag * phase_factors) @ evecs.conj().T

        # Expectation value of polarization
        P_t[idx] = np.real(np.trace(rho_t @ O_mu))

        # Partial trace for reduced density matrices
        rho_tensor = rho_t.reshape(dim_mu, dim_env, dim_mu, dim_env)
        rho_mu = np.trace(rho_tensor, axis1=1, axis2=3)
        rho_F = np.trace(rho_tensor, axis1=0, axis2=2)

        S_mu[idx] = von_neumann_entropy(rho_mu)
        # S_F(t) - S_F(0) offset normalization as in Paper Figure 2
        S_F[idx] = von_neumann_entropy(rho_F) - np.log2(dim_env)

    return P_t, S_mu, S_F


# --- Simulation Run ---
tlist = np.linspace(0, 20e-6, 300)  # 0 to 20 microseconds
tlist_us = tlist * 1e6

# Calculate for F-mu-F + 8 NNN fluorines (s_mu || z)
cluster_nnn = build_f_mu_f_cluster(include_nnn=True)
Pz, Smu, SF = compute_signals_and_entropy(
    cluster_nnn, tlist, muon_init_axis="z"
)

# --- Plotting Results (Matching Fig. 2 of PRL paper) ---
fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True)

axes[0].plot(tlist_us, Pz, "k-", lw=1.8, label=r"$P_z^\mu(t)$")
axes[0].set_ylabel(r"$P_z^\mu(t)$", fontsize=12)
axes[0].set_ylim(-0.1, 1.1)

axes[1].plot(tlist_us, Smu, "b-", lw=1.8, label=r"$S_\mu(t)$")
axes[1].set_ylabel(r"$S_\mu(t)$ [bits]", fontsize=12)
axes[1].set_ylim(-0.05, 1.1)

axes[2].plot(tlist_us, SF, "r-", lw=1.8, label=r"$S_F(t) - S_F(0)$")
axes[2].set_ylabel(r"$S_F(t) - S_F(0)$", fontsize=12)
axes[2].set_xlabel(r"$t\ (\mu\mathrm{s})$", fontsize=12)

for ax in axes:
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

AttributeError: 'MuonNuclearInteraction' object has no attribute 'H'